# Tutorial 05: Global N-gram Registry — Universe Sets for the HLLSet Framework

This tutorial introduces the `GlobalNGramRegistry` module, which maintains persistent global HLLSets that accumulate ALL observed n-grams across the system lifetime.

## What You'll Learn

1. **Universe Sets** — G₁, G₂, G₃ for unigrams, bigrams, trigrams
2. **N-gram Ingestion** — Accumulating tokens into layers
3. **Complement Operations** — Finding what's NOT in a set
4. **Coverage & Novelty** — Measuring set relationships to the universe
5. **Snapshot & Restore** — Persisting registry state
6. **Registry Merging** — Combining registries from different sources

## Key Concepts

The registry maintains universe sets:

$$G_1 = \bigcup_t \text{unigrams}(t) \quad \text{(all 1-grams ever seen)}$$
$$G_2 = \bigcup_t \text{bigrams}(t) \quad \text{(all 2-grams ever seen)}$$
$$G_3 = \bigcup_t \text{trigrams}(t) \quad \text{(all 3-grams ever seen)}$$

These serve as:
- **Universe for complement**: $\neg A = G_k \setminus A$
- **Membership tests**: "Has this n-gram been observed?"
- **Normalization for BSS**: Coverage of vocabulary
- **Bloom-filter-like filtering**: Fast pre-filtering before disambiguation

In [ ]:
# Import modules
import sys
sys.path.insert(0, '..')

from core.global_registry import (
    GlobalNGramRegistry, 
    RegistrySnapshot,
    DEFAULT_MAX_N,
    LAYER_NAMES
)
from core.hllset import HLLSet

## 1. Creating a Registry

The `GlobalNGramRegistry` is an **explicit object** — no module-level singletons. Applications create it, populate it, and pass it to components.

In [ ]:
# Create an empty registry
registry = GlobalNGramRegistry(p_bits=10, max_n=3)

print(f"Registry: {registry}")
print(f"Precision bits: {registry.p_bits}")
print(f"Max n-gram order: {registry.max_n}")
print(f"\nLayer names:")
for layer in range(registry.max_n):
    print(f"  Layer {layer}: {registry.layer_name(layer)}")

In [ ]:
# Initially, all universe sets are empty
print("Initial cardinalities:")
for layer, card in registry.all_cardinalities().items():
    print(f"  G{layer+1} ({registry.layer_name(layer)}): ~{card:.0f}")

## 2. Ingesting Tokens

When you ingest a token sequence, the registry extracts ALL n-grams (1-gram through max_n-gram) and accumulates them into the corresponding layers.

For sequence `[t₁, t₂, t₃, t₄]`:
- **Layer 0 (unigrams)**: `{t₁}, {t₂}, {t₃}, {t₄}`
- **Layer 1 (bigrams)**: `{t₁ t₂}, {t₂ t₃}, {t₃ t₄}`
- **Layer 2 (trigrams)**: `{t₁ t₂ t₃}, {t₂ t₃ t₄}`

In [ ]:
# Ingest a token sequence
tokens = ["the", "quick", "brown", "fox", "jumps"]
counts = registry.ingest(tokens)

print(f"Ingested tokens: {tokens}")
print(f"\nN-grams extracted per layer:")
for layer, count in counts.items():
    print(f"  Layer {layer} ({registry.layer_name(layer)}): {count} n-grams")

In [ ]:
# Check updated cardinalities
print("Updated cardinalities:")
for layer, card in registry.all_cardinalities().items():
    print(f"  G{layer+1}: ~{card:.0f}")

In [ ]:
# Ingest more tokens — registry accumulates (union)
more_tokens = ["the", "lazy", "dog", "sleeps"]  # "the" overlaps
counts2 = registry.ingest(more_tokens)

print(f"Ingested: {more_tokens}")
print(f"\nN-grams extracted: {counts2}")
print(f"\nUpdated G₁ cardinality: ~{registry.layer_cardinality(0):.0f}")
print(f"(Note: 'the' appears in both, but counted once due to HLL union)")

In [ ]:
# Ingest a document (splits on whitespace)
document = "machine learning is transforming artificial intelligence research"
doc_counts = registry.ingest_document(document)

print(f"Ingested document: '{document[:40]}...'")
print(f"N-grams per layer: {doc_counts}")

In [ ]:
# Batch ingest multiple documents
documents = [
    "deep learning neural networks",
    "natural language processing models",
    "computer vision algorithms",
]

batch_counts = registry.ingest_batch_documents(documents)
print(f"Batch ingested {len(documents)} documents")
print(f"Total n-grams per layer: {batch_counts}")

## 3. Universe Operations

The registry provides access to universe sets $G_k$ for each layer.

In [ ]:
# Access universe sets
G1 = registry.universe(layer=0)  # Unigrams
G2 = registry.universe(layer=1)  # Bigrams
G3 = registry.universe(layer=2)  # Trigrams

print(f"Universe set cardinalities:")
print(f"  G₁ (unigrams):  ~{G1.cardinality():.0f}")
print(f"  G₂ (bigrams):   ~{G2.cardinality():.0f}")
print(f"  G₃ (trigrams):  ~{G3.cardinality():.0f}")

In [ ]:
# Universe sets are regular HLLSets — all operations work
print(f"G₁ type: {type(G1).__name__}")
print(f"G₁ name (SHA1): {G1.name[:16]}...")
print(f"G₁ p_bits: {G1.p_bits}")

## 4. Complement Operations

The complement of a set $A$ relative to universe $G_k$:

$$\neg A = G_k \setminus A$$

This gives "all n-grams in the universe that are NOT in A".

In [ ]:
# Create a subset HLLSet
my_tokens = ["machine", "learning", "deep", "neural"]
my_hll = HLLSet.from_batch(my_tokens)

print(f"My set: {my_tokens}")
print(f"My set cardinality: ~{my_hll.cardinality():.0f}")

In [ ]:
# Compute complement: G₁ \ my_hll
complement = registry.complement(my_hll, layer=0)

print(f"Complement (G₁ \\ my_set):")
print(f"  G₁ cardinality:         ~{G1.cardinality():.0f}")
print(f"  My set cardinality:     ~{my_hll.cardinality():.0f}")
print(f"  Complement cardinality: ~{complement.cardinality():.0f}")

## 5. Coverage & Novelty

Two important metrics measure how a set relates to the universe:

- **Coverage**: $|A \cap G_k| / |G_k|$ — fraction of universe covered by A
- **Novelty**: $|A \setminus G_k| / |A|$ — fraction of A that's new (not in universe)

In [ ]:
# Coverage: how much of the universe does my_hll cover?
coverage = registry.coverage(my_hll, layer=0)
print(f"Coverage of G₁ by my set: {coverage:.1%}")

In [ ]:
# Novelty: how much of a new set is NOT in the universe?
new_tokens = ["quantum", "computing", "machine", "learning"]  # 2 known, 2 new
new_hll = HLLSet.from_batch(new_tokens)

novelty = registry.novelty(new_hll, layer=0)
print(f"New tokens: {new_tokens}")
print(f"Novelty ratio: {novelty:.1%}")
print(f"(Expected ~50% since 'quantum' and 'computing' are likely new)")

In [ ]:
# High coverage + low novelty = familiar content
# Low coverage + high novelty = mostly new content

familiar = HLLSet.from_batch(["machine", "learning", "deep"])  # All known
unknown = HLLSet.from_batch(["xyz123", "abc456", "def789"])   # All new

print("Familiar content (all known tokens):")
print(f"  Coverage: {registry.coverage(familiar, 0):.1%}")
print(f"  Novelty:  {registry.novelty(familiar, 0):.1%}")

print("\nUnknown content (all new tokens):")
print(f"  Coverage: {registry.coverage(unknown, 0):.1%}")
print(f"  Novelty:  {registry.novelty(unknown, 0):.1%}")

## 6. Registry Statistics

Get comprehensive statistics about the registry state.

In [ ]:
# Get full statistics
stats = registry.stats()

print(f"Registry Statistics:")
print(f"  p_bits: {stats['p_bits']}")
print(f"  max_n: {stats['max_n']}")
print(f"\nLayer details:")
for layer, info in stats['layers'].items():
    print(f"  {info['name'].capitalize()}:")
    print(f"    Cardinality: ~{info['cardinality']:.0f}")
    print(f"    Ingested: {info['ingest_count']} n-grams")
    print(f"    SHA1: {info['sha1']}...")

## 7. Snapshot & Restore

The registry can be serialized to a snapshot and restored later. This enables:
- Persisting registry state to disk
- Sharing registries between processes
- Checkpointing for fault tolerance

In [ ]:
# Create a snapshot
snapshot = registry.snapshot()

print(f"Snapshot created:")
print(f"  Registry ID: {snapshot.registry_id[:16]}...")
print(f"  p_bits: {snapshot.p_bits}")
print(f"  Layers: {len(snapshot.layer_registers)}")
print(f"  Token counts: {snapshot.token_counts}")

In [ ]:
# Snapshot can be serialized to JSON
import json

snapshot_dict = snapshot.to_dict()
print("Snapshot as JSON (truncated):")
print(f"  registry_id: {snapshot_dict['registry_id'][:16]}...")
print(f"  p_bits: {snapshot_dict['p_bits']}")
print(f"  Layer 0 registers (first 32 hex chars): {snapshot_dict['layer_registers']['0'][:32]}...")

In [ ]:
# Restore from snapshot
restored = GlobalNGramRegistry.from_snapshot(snapshot)

print(f"Restored registry: {restored}")
print(f"\nOriginal G₁ cardinality: ~{registry.layer_cardinality(0):.0f}")
print(f"Restored G₁ cardinality: ~{restored.layer_cardinality(0):.0f}")

In [ ]:
# Verify restoration by checking coverage
test_hll = HLLSet.from_batch(["machine", "learning"])

orig_coverage = registry.coverage(test_hll, 0)
rest_coverage = restored.coverage(test_hll, 0)

print(f"Coverage test:")
print(f"  Original: {orig_coverage:.4f}")
print(f"  Restored: {rest_coverage:.4f}")
print(f"  Match: {abs(orig_coverage - rest_coverage) < 0.001}")

## 8. Registry Merging

Multiple registries can be merged (union of all layers). This is useful for:
- Combining registries from distributed processing
- Merging domain-specific vocabularies
- Aggregating results from parallel ingestion

In [ ]:
# Create two separate registries
registry_A = GlobalNGramRegistry(p_bits=10)
registry_B = GlobalNGramRegistry(p_bits=10)

# Populate with different domains
registry_A.ingest_document("machine learning deep neural networks")
registry_B.ingest_document("natural language processing text analysis")

print(f"Registry A: {registry_A}")
print(f"Registry B: {registry_B}")

In [ ]:
# Merge registries (creates new, doesn't modify originals)
merged = registry_A.merge(registry_B)

print(f"Merged registry: {merged}")
print(f"\nCardinality comparison:")
print(f"  A G₁: ~{registry_A.layer_cardinality(0):.0f}")
print(f"  B G₁: ~{registry_B.layer_cardinality(0):.0f}")
print(f"  Merged G₁: ~{merged.layer_cardinality(0):.0f}")

In [ ]:
# Originals are unchanged (immutability)
print(f"\nOriginal A unchanged: {registry_A.layer_cardinality(0):.0f}")
print(f"Original B unchanged: {registry_B.layer_cardinality(0):.0f}")

## 9. Practical Example: Document Corpus Vocabulary

Let's build a vocabulary registry from a document corpus and use it to analyze new documents.

In [ ]:
# Create a corpus registry
corpus_registry = GlobalNGramRegistry(p_bits=10, max_n=3)

# Simulate a corpus of ML papers
corpus = [
    "deep learning neural networks achieve state of the art results",
    "transformer models use attention mechanisms for sequence processing",
    "convolutional neural networks excel at image recognition tasks",
    "recurrent neural networks model sequential data effectively",
    "generative adversarial networks create realistic synthetic data",
    "reinforcement learning agents learn through trial and error",
    "natural language processing enables machine understanding of text",
    "computer vision algorithms detect objects in images and video",
]

# Ingest entire corpus
corpus_registry.ingest_batch_documents(corpus)
print(f"Corpus registry: {corpus_registry}")
print(f"\nVocabulary sizes:")
for layer in range(3):
    print(f"  {corpus_registry.layer_name(layer)}: ~{corpus_registry.layer_cardinality(layer):.0f}")

In [ ]:
# Analyze a new document
new_doc = "quantum computing will revolutionize machine learning algorithms"
new_tokens = new_doc.split()
new_hll = HLLSet.from_batch(new_tokens)

print(f"New document: '{new_doc}'")
print(f"\nAnalysis against corpus vocabulary:")
print(f"  Coverage (unigrams): {corpus_registry.coverage(new_hll, 0):.1%}")
print(f"  Novelty (unigrams):  {corpus_registry.novelty(new_hll, 0):.1%}")

In [ ]:
# Find novel tokens (approximate via complement)
novel_hll = new_hll.diff(corpus_registry.universe(0))
print(f"\nNovel token count: ~{novel_hll.cardinality():.0f}")
print(f"(Tokens like 'quantum', 'computing', 'revolutionize' are likely new)")

In [ ]:
# Compare two new documents
doc_familiar = "neural networks use attention for deep learning"
doc_novel = "quantum entanglement enables faster computation"

hll_familiar = HLLSet.from_batch(doc_familiar.split())
hll_novel = HLLSet.from_batch(doc_novel.split())

print("Document comparison:")
print(f"\nFamiliar: '{doc_familiar}'")
print(f"  Coverage: {corpus_registry.coverage(hll_familiar, 0):.1%}")
print(f"  Novelty:  {corpus_registry.novelty(hll_familiar, 0):.1%}")

print(f"\nNovel: '{doc_novel}'")
print(f"  Coverage: {corpus_registry.coverage(hll_novel, 0):.1%}")
print(f"  Novelty:  {corpus_registry.novelty(hll_novel, 0):.1%}")

## 10. Multi-Layer Analysis

The registry maintains separate universe sets for each n-gram layer, enabling multi-resolution analysis.

In [ ]:
# Analyze bigrams and trigrams of a phrase
phrase = "deep learning neural networks"
tokens = phrase.split()

# Create HLLSets for each layer
unigrams = HLLSet.from_batch(tokens)
bigrams = HLLSet.from_batch([" ".join(tokens[i:i+2]) for i in range(len(tokens)-1)])
trigrams = HLLSet.from_batch([" ".join(tokens[i:i+3]) for i in range(len(tokens)-2)])

print(f"Phrase: '{phrase}'")
print(f"\nExtracted n-grams:")
print(f"  Unigrams: {tokens}")
print(f"  Bigrams: {[' '.join(tokens[i:i+2]) for i in range(len(tokens)-1)]}")
print(f"  Trigrams: {[' '.join(tokens[i:i+3]) for i in range(len(tokens)-2)]}")

print(f"\nCoverage per layer:")
print(f"  Unigrams: {corpus_registry.coverage(unigrams, 0):.1%}")
print(f"  Bigrams:  {corpus_registry.coverage(bigrams, 1):.1%}")
print(f"  Trigrams: {corpus_registry.coverage(trigrams, 2):.1%}")

## Summary

In this tutorial, you learned:

1. **Universe Sets** — G₁, G₂, G₃ accumulate all observed n-grams
2. **Ingestion** — `ingest()`, `ingest_document()`, `ingest_batch_documents()`
3. **Complement** — $G_k \setminus A$ gives "not in A"
4. **Coverage & Novelty** — Measure set relationships to universe
5. **Snapshot/Restore** — Persist and recover registry state
6. **Merging** — Combine registries from different sources

### Key Design Points

- **Explicit object** — No global singletons; applications manage registry lifecycle
- **Union-only accumulation** — Layers only grow (monotonic)
- **Content-addressed** — Snapshot includes SHA1 for verification
- **Immutable merge** — Returns new registry, doesn't modify inputs

### Next Steps

- **Tutorial 06**: RingTransaction — IICA-compliant HLLSet construction
- **Tutorial 07**: BSS — Bell State Similarity for morphism testing